In [1]:
import os
import sys
module_path = os.path.join("./KeyPoint-Analysis/KPG/")
sys.path.insert(0, module_path)
from rouge_setbase import preprocess_dataset, compute_rouge, compute_rouge_max
from softF1 import softevaluation
# root_dir_pth = "../"

import pandas as pd
import re
import ast

2026-01-13 02:12:02.421925: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 02:12:02.445832: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 02:12:02.445860: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 02:12:02.445874: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 02:12:02.451489: I tensorflow/core/platform/cpu_feature_g

# Read Data

In [2]:
ground_truth_df = pd.read_pickle(f"../data/test/test.pkl")
ground_truth_df = ground_truth_df[['category', 'product_name', 'user_id', 's5_annotated_personalized_summaries']]
ground_truth_df = ground_truth_df.rename(columns={'s5_annotated_personalized_summaries': 'key_point_given'})
ground_truth_df

,category,product_name,user_id,key_point_given
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,If you’re tackling dandruff or even dealing wi...
1,Beauty,Gillette Mach 3 Razor,5000858,"Alright, fellow grooming adventurers, let’s ta..."
2,Beauty,Pitrok,5296801,If you’re on the lookout for a natural deodora...
3,Beauty,Gillette Mach 3 Razor,5050855,"If you’re weighing the Gillette Mach 3 razor, ..."
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,If you’re on the lookout for a budget-friendly...
...,...,...,...,...
95,Travel,Dollar Rent A Car Worldwide,5297771,"If you’re a budget-savvy traveler like me, Dol..."
96,Travel,Amsterdam (Netherlands),5202501,"If you’re planning a trip to Amsterdam, here’s..."
97,Travel,Leicester in General,5020891,Leicester might not top the usual tourist list...
98,Travel,Milan in general,5091015,"If you’re planning a trip to Milan, here’s a b..."


In [3]:
root_path = f"../output/stage_2_rl_inference/summary_kp_extraction"
temp_df = pd.read_pickle(root_path + "/1/1_done.pkl")
temp_df = temp_df.rename(columns={'voter_full': 'user_id'})
temp_df.shape

(96, 9)

In [4]:
mask = pd.isnull(temp_df['personalized_summaries'])
temp_df = temp_df[~mask]
temp_df.shape

(96, 9)

In [5]:
temp_df['personalized_summaries'] = temp_df['personalized_summaries'].apply(lambda x: re.sub(r"Here[^\:]+\:", "", x).strip())
temp_df['personalized_summaries'] = temp_df['personalized_summaries'].apply(lambda x: re.sub(r"Note[^\:]+\:", "", x).strip())
temp_df['personalized_summaries'] = temp_df['personalized_summaries'].apply(lambda x: re.sub(r"Base[^\:]+\:", "", x).strip())
temp_df['personalized_summaries'] = temp_df['personalized_summaries'].apply(lambda x: "\n\n".join(x.split("\n\n")[:2]))

In [6]:
print(temp_df['personalized_summaries'].iloc[1])

The Gillette Mach 3 Razor is a high-quality razor that provides a close and comfortable shave. With its three blades and lubricating strip, it is easy to use and provides a smooth shave. The razor is durable and the blades last a long time, making it a good value for the price. The razor is also easy to clean and maintain, making it a great choice for those who want a hassle-free shaving experience.

As a user who is particular about getting a close shave, you will appreciate the effectiveness of the Mach 3 Razor. The razor is designed to provide a close shave with minimal irritation, making it a great choice for those with sensitive skin. The razor is also easy to use, with a comfortable grip and a simple design that makes it easy to maneuver.


In [7]:
mask = temp_df['personalized_summaries'].apply(lambda x: re.findall(r"\[.+\]", x)).str.len() > 0
temp_df['personalized_summaries'] = temp_df['personalized_summaries'].apply(lambda x: re.sub(r"\[.+\]", "", x).strip())

In [8]:
temp_df = temp_df.rename(columns={'personalized_summaries': 'key_point'})

In [9]:
claim_split_predicted = temp_df.merge(ground_truth_df)

In [10]:
claim_split_predicted

,claim_split_predicted,key_point,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,"[\n ""Head & Shoulders Normal Hair Shampoo is ...",**Effective Dandruff Control**\n\nHead & Shoul...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,If you’re tackling dandruff or even dealing wi...
1,"```json\n[\n ""The Gillette Mach 3 Razor provi...",The Gillette Mach 3 Razor is a high-quality ra...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1,"Alright, fellow grooming adventurers, let’s ta..."
2,"[\n ""Pitrok is a natural deodorant free from ...",As a health-conscious individual who values na...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1,If you’re on the lookout for a natural deodora...
3,"```json\n[\n ""The Gillette Mach 3 Razor deliv...","As a seasoned shaver, you know that a good raz...",Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1,"If you’re weighing the Gillette Mach 3 razor, ..."
4,"```json\n[\n ""The shampoo is gentle and effec...","If you're looking for a shampoo that's gentle,...",Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1,If you’re on the lookout for a budget-friendly...
...,...,...,...,...,...,...,...,...,...,...
91,"[\n ""Dollar Rent A Car Worldwide is a reliabl...",Dollar Rent A Car Worldwide is a reliable and ...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1,"If you’re a budget-savvy traveler like me, Dol..."
92,"[\n ""Amsterdam is steeped in tradition and is...",Amsterdam is a city that is sure to surprise a...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1,"If you’re planning a trip to Amsterdam, here’s..."
93,"[\n ""Leicester offers a diverse range of shop...",Leicester is a city that truly has something f...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1,Leicester might not top the usual tourist list...
94,"```json\n[\n ""Milan features stunning archite...",Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,"If you’re planning a trip to Milan, here’s a b..."


In [11]:
merged_df = claim_split_predicted.explode(['key_point']).explode(['key_point_given'])

In [12]:
merged_df

,claim_split_predicted,key_point,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,"[\n ""Head & Shoulders Normal Hair Shampoo is ...",**Effective Dandruff Control**\n\nHead & Shoul...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,If you’re tackling dandruff or even dealing wi...
1,"```json\n[\n ""The Gillette Mach 3 Razor provi...",The Gillette Mach 3 Razor is a high-quality ra...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1,"Alright, fellow grooming adventurers, let’s ta..."
2,"[\n ""Pitrok is a natural deodorant free from ...",As a health-conscious individual who values na...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1,If you’re on the lookout for a natural deodora...
3,"```json\n[\n ""The Gillette Mach 3 Razor deliv...","As a seasoned shaver, you know that a good raz...",Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1,"If you’re weighing the Gillette Mach 3 razor, ..."
4,"```json\n[\n ""The shampoo is gentle and effec...","If you're looking for a shampoo that's gentle,...",Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1,If you’re on the lookout for a budget-friendly...
...,...,...,...,...,...,...,...,...,...,...
91,"[\n ""Dollar Rent A Car Worldwide is a reliabl...",Dollar Rent A Car Worldwide is a reliable and ...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1,"If you’re a budget-savvy traveler like me, Dol..."
92,"[\n ""Amsterdam is steeped in tradition and is...",Amsterdam is a city that is sure to surprise a...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1,"If you’re planning a trip to Amsterdam, here’s..."
93,"[\n ""Leicester offers a diverse range of shop...",Leicester is a city that truly has something f...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1,Leicester might not top the usual tourist list...
94,"```json\n[\n ""Milan features stunning archite...",Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,"If you’re planning a trip to Milan, here’s a b..."


# Evaluation

In [13]:
from softF1 import *

## BERTScore

In [14]:
softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\
    .apply(lambda grp: "".join([cand + "=" for cand in (grp['key_point_given'].tolist())])).reset_index(name='multi_cands')
softp_data

/tmp/ipykernel_2440/1500243889.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\


,category,product_name,user_id,key_point,multi_cands
0,Beauty,Gillette Mach 3 Razor,5000858,The Gillette Mach 3 Razor is a high-quality ra...,"Alright, fellow grooming adventurers, let’s ta..."
1,Beauty,Gillette Mach 3 Razor,5050855,"As a seasoned shaver, you know that a good raz...","If you’re weighing the Gillette Mach 3 razor, ..."
2,Beauty,Head & Shoulders Normal Hair Shampoo,3680,**Effective Dandruff Control**\n\nHead & Shoul...,If you’re tackling dandruff or even dealing wi...
3,Beauty,Pitrok,5296801,As a health-conscious individual who values na...,If you’re on the lookout for a natural deodora...
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,"If you're looking for a shampoo that's gentle,...",If you’re on the lookout for a budget-friendly...
...,...,...,...,...,...
91,Travel,Amsterdam (Netherlands),5202501,Amsterdam is a city that is sure to surprise a...,"If you’re planning a trip to Amsterdam, here’s..."
92,Travel,Budapest (Hungary),5116809,Budapest is a city that will leave you enchant...,If you’re considering Budapest for your next a...
93,Travel,Dollar Rent A Car Worldwide,5297771,Dollar Rent A Car Worldwide is a reliable and ...,"If you’re a budget-savvy traveler like me, Dol..."
94,Travel,Leicester in General,5020891,Leicester is a city that truly has something f...,Leicester might not top the usual tourist list...


In [15]:
cands, refs= preprocess_text(softp_data, metrics = "softPrecision")

P, R, F = bert_scorer.score(cands, refs)
P_average = P.mean()

In [16]:
P_average

tensor(0.2481)

## BARTScore

In [17]:
softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\
    .apply(lambda grp: "".join([cand + "=" for cand in (grp['key_point_given'].tolist())])).reset_index(name='multi_cands')
softp_data

/tmp/ipykernel_2255/1500243889.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\


,category,product_name,user_id,key_point,multi_cands
0,Beauty,Gillette Mach 3 Razor,5000858,Based on the user profile and the helpful key ...,"Alright, fellow grooming adventurers, let’s ta..."
1,Beauty,Gillette Mach 3 Razor,5050855,Here is a personalized summary of product A (G...,"If you’re weighing the Gillette Mach 3 razor, ..."
2,Beauty,Head & Shoulders Normal Hair Shampoo,3680,Here is a personalized summary of product A (H...,If you’re tackling dandruff or even dealing wi...
3,Beauty,Pitrok,5296801,Based on the helpful key points and the user p...,If you’re on the lookout for a natural deodora...
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,Based on the user profile and the helpful key ...,If you’re on the lookout for a budget-friendly...
...,...,...,...,...,...
91,Travel,Amsterdam (Netherlands),5202501,Here is a personalized summary of product A (A...,"If you’re planning a trip to Amsterdam, here’s..."
92,Travel,Budapest (Hungary),5116809,Here is a personalized summary of Budapest tai...,If you’re considering Budapest for your next a...
93,Travel,Dollar Rent A Car Worldwide,5297771,Based on the helpful key points and user 111's...,"If you’re a budget-savvy traveler like me, Dol..."
94,Travel,Leicester in General,5020891,Here is a personalized summary of product A (L...,Leicester might not top the usual tourist list...


In [18]:
cands, refs= preprocess_text(softp_data, metrics = "softPrecision")

#BART score cannot be processed for different numbers of reference sentences, so
#check for the maximum number of reference sentences and match the size.
#We fill in the None for the missing sentences because we only pick one of the 
#maximum values and not the average, so there is no impact on performance

refs = balance_ref_num(refs)        

# generation scores from the first list of texts to the second list of texts.
P = bart_scorer.multi_ref_score(cands, refs, agg="max", batch_size=4) # agg means aggregation, can be mean or max

#mapping the score to (0,1]
P_average = math.tanh(math.exp((mean(P))/2+1.3))

In [19]:
P_average

0.5764490150478356

## BLEURTScore

In [20]:
softp_data = merged_df.sort_values(by=['category', 'product_name', 'user_id', 'key_point'])
df_compare_precision = softp_data[['key_point', 'key_point_given']].rename(columns={'key_point': 'candidate', 'key_point_given': 'reference'})
df_compare_precision

,candidate,reference
1,The Gillette Mach 3 Razor is a high-quality ra...,"Alright, fellow grooming adventurers, let’s ta..."
3,"As a seasoned shaver, you know that a good raz...","If you’re weighing the Gillette Mach 3 razor, ..."
0,**Effective Dandruff Control**\n\nHead & Shoul...,If you’re tackling dandruff or even dealing wi...
2,As a health-conscious individual who values na...,If you’re on the lookout for a natural deodora...
4,"If you're looking for a shampoo that's gentle,...",If you’re on the lookout for a budget-friendly...
...,...,...
92,Amsterdam is a city that is sure to surprise a...,"If you’re planning a trip to Amsterdam, here’s..."
95,Budapest is a city that will leave you enchant...,If you’re considering Budapest for your next a...
91,Dollar Rent A Car Worldwide is a reliable and ...,"If you’re a budget-savvy traveler like me, Dol..."
93,Leicester is a city that truly has something f...,Leicester might not top the usual tourist list...


In [21]:
candidates = df_compare_precision['candidate']
references = df_compare_precision['reference']

In [22]:
import tensorflow as tf

In [23]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


2026-01-13 02:13:03.737105: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:03.741058: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:03.741443: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [24]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 16974871924855605872
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 16651386880
locality {
  bus_id: 1
  links {
  }
}
incarnation: 7005142408071351907
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


2026-01-13 02:13:03.896145: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:03.896561: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:03.897054: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:03.898103: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:03.898423: I tensorflow/compile

In [25]:
# with tf.device('/gpu:0'):
#   result = calculatingScore(references, candidates)
#   df_compare_precision["BLEURT Score"] = result

#   #After calculating the semantic quality of all candidates and reference pairs, the one with the highest score is selected as the correct pair.
#   df_bestkp_pair_precision = df_compare_precision.loc[df_compare_precision.groupby(["candidate"])["BLEURT Score"].idxmax()]
#   #take average of all best scores as the soft precision score.
#   P_average = df_bestkp_pair_precision["BLEURT Score"].mean()

In [26]:
softp_data = merged_df.sort_values(by=['category', 'product_name', 'user_id', 'key_point'])
df_compare_precision = softp_data[['key_point', 'key_point_given']].rename(columns={'key_point': 'candidate', 'key_point_given': 'reference'})
df_compare_precision

,candidate,reference
1,The Gillette Mach 3 Razor is a high-quality ra...,"Alright, fellow grooming adventurers, let’s ta..."
3,"As a seasoned shaver, you know that a good raz...","If you’re weighing the Gillette Mach 3 razor, ..."
0,**Effective Dandruff Control**\n\nHead & Shoul...,If you’re tackling dandruff or even dealing wi...
2,As a health-conscious individual who values na...,If you’re on the lookout for a natural deodora...
4,"If you're looking for a shampoo that's gentle,...",If you’re on the lookout for a budget-friendly...
...,...,...
92,Amsterdam is a city that is sure to surprise a...,"If you’re planning a trip to Amsterdam, here’s..."
95,Budapest is a city that will leave you enchant...,If you’re considering Budapest for your next a...
91,Dollar Rent A Car Worldwide is a reliable and ...,"If you’re a budget-savvy traveler like me, Dol..."
93,Leicester is a city that truly has something f...,Leicester might not top the usual tourist list...


In [27]:
candidates = df_compare_precision['candidate']
references = df_compare_precision['reference']

In [28]:
import tensorflow as tf

In [29]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [30]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 4616815096512661812
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 16651386880
locality {
  bus_id: 1
  links {
  }
}
incarnation: 13844124780744872723
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


2026-01-13 02:13:05.896941: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:05.897543: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:05.898162: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:05.898924: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:05.898936: I tensorflow/core/co

In [31]:
result = calculatingScore(references, candidates)
df_compare_precision["BLEURT Score"] = result

#After calculating the semantic quality of all candidates and reference pairs, the one with the highest score is selected as the correct pair.
df_bestkp_pair_precision = df_compare_precision.loc[df_compare_precision.groupby(["candidate"])["BLEURT Score"].idxmax()]
#take average of all best scores as the soft precision score.
P_average = df_bestkp_pair_precision["BLEURT Score"].mean()

INFO:tensorflow:Reading checkpoint /mnt/d/Desktop/PHD READING/HELPFULSUMM/evaluation/KeyPoint-Analysis/KPG/BLEURT-20.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: /mnt/d/Desktop/PHD READING/HELPFULSUMM/evaluation/KeyPoint-Analysis/KPG/BLEURT-20/sent_piece.model.
INFO:tensorflow:SentencePiece tokenizer created.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.


2026-01-13 02:13:06.809923: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:06.810410: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:06.810664: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:06.811130: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:13:06.811138: I tensorflow/core/co

INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [32]:
P_average

0.47675184595088166

## ROUGE

In [33]:
gt_gold_kp = merged_df

In [34]:
gt_gold_kp

,claim_split_predicted,key_point,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,"[\n ""Head & Shoulders Normal Hair Shampoo is ...",Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,If you’re tackling dandruff or even dealing wi...
1,"```json\n[\n ""The Gillette Mach 3 Razor provi...",Based on the user profile and the helpful key ...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1,"Alright, fellow grooming adventurers, let’s ta..."
2,"[\n ""Pitrok is a natural deodorant free from ...",Based on the helpful key points and the user p...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1,If you’re on the lookout for a natural deodora...
3,"```json\n[\n ""The Gillette Mach 3 Razor deliv...",Here is a personalized summary of product A (G...,Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1,"If you’re weighing the Gillette Mach 3 razor, ..."
4,"```json\n[\n ""The shampoo is gentle and effec...",Based on the user profile and the helpful key ...,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1,If you’re on the lookout for a budget-friendly...
...,...,...,...,...,...,...,...,...,...,...
91,"[\n ""Dollar Rent A Car Worldwide is a reliabl...",Based on the helpful key points and user 111's...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1,"If you’re a budget-savvy traveler like me, Dol..."
92,"[\n ""Amsterdam is steeped in tradition and is...",Here is a personalized summary of product A (A...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1,"If you’re planning a trip to Amsterdam, here’s..."
93,"[\n ""Leicester offers a diverse range of shop...",Here is a personalized summary of product A (L...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1,Leicester might not top the usual tourist list...
94,"```json\n[\n ""Milan features stunning archite...",Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,"If you’re planning a trip to Milan, here’s a b..."


In [35]:
predictions, references = [], []
for topic in sorted(gt_gold_kp['product_name'].unique()):
    kps = gt_gold_kp.loc[(gt_gold_kp['product_name']==topic), 'key_point'].unique().tolist()
    gold_kps = gt_gold_kp.loc[(gt_gold_kp['product_name']==topic), 'key_point_given'].unique().tolist()
    if len(kps) > 0 and len(gold_kps) > 0:
        predictions.append(kps)
        references.append(gold_kps)

In [36]:
compute_rouge(predictions, references)

Rouge 1: 0.401
Rouge 2: 0.088
Rouge L: 0.177
